# Neutrino oscillations with QARP time evolution

This tutorial simulates two-flavor neutrino oscillations with QARP's symbolic `TrotterBlock`. It begins with the analytic one-neutrino problem in vacuum and constant-density matter, then adds a two-neutrino interaction and validates it against direct dense-matrix evolution.

In the flavor basis $|\nu_e\rangle=|0\rangle$, $|\nu_x\rangle=|1\rangle$, use the dimensionless Hamiltonian

$$
H_i=\omega_i\sin(2\theta)X_i+
\left[-\omega_i\cos(2\theta)+\lambda_e\right]Z_i.
$$

For two neutrinos we add $g(X_0X_1+Y_0Y_1+Z_0Z_1)$, the two-particle form of the collective forward-scattering term discussed in the literature.

**Encoding and scope.** This uses one qubit per neutrino and prepares computational-basis flavor states with $X$ gates. There is no amplitude loading. The constant-density model is pedagogical, not a realistic supernova transport calculation, and no quantum speedup is claimed.

References: the [HEP quantum-computing review](https://arxiv.org/abs/2307.03236) and a [quantum-simulation treatment of collective neutrino oscillations](https://arxiv.org/abs/2308.09123).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm
from sympy import Symbol

from qarp.algorithms import StateVector
from qarp.blocks import CompositeBlock, ComputationalBasisStateBlock, TrotterBlock
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator
from qarp.resources import Stage, estimate

mixing_angle = 0.60
omega_0 = 1.0
omega_1 = 0.72  # Illustrative second energy mode; omega is proportional to 1 / E.
time_symbol = Symbol("t")

## 1. One neutrino: analytic benchmark

For $H=b_xX+b_zZ$, $\Omega=\sqrt{b_x^2+b_z^2}$ and an initial electron flavor has survival probability

$$
P_{ee}(t)=1-\frac{b_x^2}{\Omega^2}\sin^2(\Omega t).
$$

The QARP circuit is built once with a symbolic time and reused throughout each scan. A second-order, 32-step product formula makes the Trotter error visible but small over this deliberately long interval.

In [ ]:
def one_neutrino_hamiltonian(omega_value, matter_potential):
    x_coefficient = omega_value * np.sin(2 * mixing_angle)
    z_coefficient = -omega_value * np.cos(2 * mixing_angle) + matter_potential
    hamiltonian = QubitOperator("X0", x_coefficient)
    hamiltonian += QubitOperator("Z0", z_coefficient)
    return hamiltonian, x_coefficient, z_coefficient


def survival_scan(matter_potential, times):
    hamiltonian, x_coefficient, z_coefficient = one_neutrino_hamiltonian(omega_0, matter_potential)
    state = CompositeBlock(
        [
            ComputationalBasisStateBlock([0]),
            TrotterBlock(
                n_qubits=1,
                operator=hamiltonian,
                steps=32,
                time=time_symbol,
                order=2,
            ),
        ]
    ).build()
    z_expectation = StateVector(bra=state, operator=QubitOperator("Z0"), ket=state)
    engine = QarpEngine()
    engine.build([z_expectation])
    qarp_probability = np.array(
        [(1.0 + float(engine.run({time_symbol: value})[0].real)) / 2.0 for value in times]
    )

    frequency = np.hypot(x_coefficient, z_coefficient)
    analytic_probability = 1.0 - (x_coefficient / frequency) ** 2 * np.sin(frequency * times) ** 2
    return qarp_probability, analytic_probability

In [ ]:
times = np.linspace(0.0, 8.0, 81)
resonance_potential = omega_0 * np.cos(2 * mixing_angle)

vacuum_qarp, vacuum_exact = survival_scan(0.0, times)
matter_qarp, matter_exact = survival_scan(resonance_potential, times)

vacuum_error = np.max(np.abs(vacuum_qarp - vacuum_exact))
matter_error = np.max(np.abs(matter_qarp - matter_exact))
print(f"Maximum vacuum error: {vacuum_error:.3e}")
print(f"Maximum resonant-matter error: {matter_error:.3e}")
assert vacuum_error < 8e-3
assert matter_error < 1e-12

fig, axes = plt.subplots(1, 2, figsize=(12, 3.7), sharey=True)
for ax, qarp_values, exact_values, title in [
    (axes[0], vacuum_qarp, vacuum_exact, "Vacuum"),
    (axes[1], matter_qarp, matter_exact, "Constant matter at resonance"),
]:
    ax.plot(times, exact_values, color="black", label="analytic")
    ax.scatter(times[::4], qarp_values[::4], s=20, color="tab:blue", label="QARP Trotter")
    ax.set(xlabel="dimensionless time", title=title)
axes[0].set_ylabel(r"$P(\nu_e\to\nu_e)$")
axes[0].legend()
plt.tight_layout()
plt.show()

At the chosen resonance, the $Z$ coefficient vanishes. The Hamiltonian then contains only $X$, so the product formula is exact; the vacuum curve retains a small accumulated splitting error.

## 2. Two interacting neutrinos

We start in $|\nu_e\rangle_0|\nu_x\rangle_1=|0,1\rangle$ (QARP tuples list $q_0$ first) and set the dimensionless vacuum frequencies to $\omega_0=1$ and $\omega_1=0.72$. The second value is an illustrative choice, not a number extracted from the paper. Since $\omega\propto 1/E$, it represents $E_1/E_0=1/0.72\simeq1.39$. We measure $\langle Z_0\rangle$, $\langle Z_1\rangle$, and the correlation $\langle Z_0Z_1\rangle$.

In [ ]:
def single_mode_terms(qubit, omega_value, matter_potential=0.0):
    return QubitOperator(f"X{qubit}", omega_value * np.sin(2 * mixing_angle)) + QubitOperator(
        f"Z{qubit}",
        -omega_value * np.cos(2 * mixing_angle) + matter_potential,
    )


coupling = 0.28
two_neutrino_hamiltonian = single_mode_terms(0, omega_0) + single_mode_terms(1, omega_1)
two_neutrino_hamiltonian += coupling * (
    QubitOperator("X0 X1") + QubitOperator("Y0 Y1") + QubitOperator("Z0 Z1")
)

two_neutrino_state = CompositeBlock(
    [
        ComputationalBasisStateBlock([0, 1]),
        TrotterBlock(
            n_qubits=2,
            operator=two_neutrino_hamiltonian,
            steps=24,
            time=time_symbol,
            order=2,
        ),
    ]
).build()

observables = [
    QubitOperator("Z0"),
    QubitOperator("Z1"),
    QubitOperator("Z0 Z1"),
]
measurements = [
    StateVector(bra=two_neutrino_state, operator=observable, ket=two_neutrino_state)
    for observable in observables
]
two_neutrino_engine = QarpEngine()
two_neutrino_engine.build(measurements)

times_two = np.linspace(0.0, 6.0, 101)
qarp_observables = np.array(
    [
        [float(value.real) for value in two_neutrino_engine.run({time_symbol: time})]
        for time in times_two
    ]
)

### Independent dense-matrix benchmark

The following reference does not call QARP's operator-to-matrix or time-evolution routines. It constructs the $4\times4$ matrix explicitly with SciPy. Since QARP is little-endian, a one-qubit operator on $q_0$ is $I\otimes P$, while one on $q_1$ is $P\otimes I$.

In [ ]:
identity = np.eye(2, dtype=complex)
pauli_x = np.array([[0, 1], [1, 0]], dtype=complex)
pauli_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
pauli_z = np.diag([1, -1]).astype(complex)


def on_qubit(matrix, qubit):
    return np.kron(identity, matrix) if qubit == 0 else np.kron(matrix, identity)


dense_hamiltonian = (
    omega_0 * np.sin(2 * mixing_angle) * on_qubit(pauli_x, 0)
    - omega_0 * np.cos(2 * mixing_angle) * on_qubit(pauli_z, 0)
    + omega_1 * np.sin(2 * mixing_angle) * on_qubit(pauli_x, 1)
    - omega_1 * np.cos(2 * mixing_angle) * on_qubit(pauli_z, 1)
    + coupling * (np.kron(pauli_x, pauli_x) + np.kron(pauli_y, pauli_y) + np.kron(pauli_z, pauli_z))
)
initial_state = np.array([0, 0, 1, 0], dtype=complex)  # |q1 q0> = |10>
dense_observables = [
    np.kron(identity, pauli_z),
    np.kron(pauli_z, identity),
    np.kron(pauli_z, pauli_z),
]

dense_results = []
for time in times_two:
    state = expm(-1j * dense_hamiltonian * time) @ initial_state
    dense_results.append(
        [np.vdot(state, observable @ state).real for observable in dense_observables]
    )
dense_results = np.asarray(dense_results)

maximum_errors = np.max(np.abs(qarp_observables - dense_results), axis=0)
print("Maximum errors for <Z0>, <Z1>, <Z0 Z1>:", maximum_errors)
assert np.all(maximum_errors < 1.8e-2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.7))
for qubit, ax in enumerate(axes):
    qarp_survival = (1.0 + qarp_observables[:, qubit]) / 2.0
    exact_survival = (1.0 + dense_results[:, qubit]) / 2.0
    ax.plot(times_two, exact_survival, color="black", label="dense reference")
    ax.scatter(times_two[::3], qarp_survival[::3], s=18, label="QARP Trotter")
    ax.set(
        xlabel="dimensionless time",
        ylabel=f"$P(\\nu_e)$ on qubit {qubit}",
        title=f"Neutrino {qubit}",
    )
axes[0].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(times_two, dense_results[:, 2], color="black", label="dense reference")
ax.scatter(times_two[::3], qarp_observables[::3, 2], s=18, label="QARP Trotter")
ax.set(xlabel="dimensionless time", ylabel=r"$\langle Z_0Z_1\rangle$", title="Flavor correlation")
ax.legend()
plt.show()

In [ ]:
logical = estimate(two_neutrino_state)[Stage.LOGICAL]
print(
    f"Two-neutrino Trotter circuit: {logical.n_qubits} qubits, "
    f"depth {logical.depth}, {logical.n_2q} two-qubit gates"
)

## Takeaways

- The analytic one-neutrino curve isolates product-formula error cleanly.
- The two-neutrino extension shows a genuinely interacting Hamiltonian and QARP's reusable symbolic parameters, multiple observables, state-vector engine, and resource estimator.
- For $N$ neutrinos the direct encoding uses $N$ qubits, but the all-to-all pair Hamiltonian has $O(N^2)$ terms. That interaction and circuit depth—not state preparation—are the scaling bottlenecks in this model.